<div dir="rtl">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ariefouren/ONO-linear-algebra-2026/blob/main/03_Chapters/03_Matrices/notebooks/03.05_invertibility_characterizations.ipynb)

# אלגברה לינארית – פרק 3: מטריצות
## מחברת 03.05: אפיונים נוספים של מטריצות הפיכות

מחברת זו מלווה את **סיכום 03.05** (סעיף 3.10; נלמד אחרי פרק 2 בספר). הספר הרשמי של הקורס: פארן וליבוביץ, *אלגברה לינארית 1, כרך א*, האוניברסיטה הפתוחה, 2019 (זמין בספרייה הדיגיטלית של הקריה האקדמית אונו, מאגר למדא).

**איך משתמשים:** בתפריט *Runtime* בוחרים *Run all*, או מריצים תא-תא עם Shift+Enter. מומלץ לשנות את המספרים בתאים ולהריץ שוב – כך רואים מה כל פעולה עושה.

הכנה: ד"ר אריה פורן, הקריה האקדמית אונו, תשפ"ז.

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(suppress=True, precision=4)

def show(name, M):
    """Print a matrix with its name and its order (rows x columns)."""
    M = np.asarray(M)
    print(f"{name}  ({M.shape[0]}x{M.shape[1] if M.ndim > 1 else 1})")
    print(M, end="\n\n")

<div dir="rtl">

## 1. המשפט המסכם (משפט 3.10.6) – בדיקה במחשב

הפונקציה בודקת כמה מהתנאים השקולים בבת אחת. **דרגה** (rank) היא מספר העמודות הבלתי תלויות; את המושג נלמד לעומק בהמשך, וכאן משתמשים בו רק ככלי בדיקה.

</div>

In [ ]:
import sympy as sp

def invertibility_report(A):
    A = sp.Matrix(A); n = A.rows
    rref, pivots = A.rref()
    rows = [
        ("(d) RREF of A is I",                    rref == sp.eye(n)),
        ("(g) Sol(A|0) = {0}",                     len(A.nullspace()) == 0),
        ("(h) columns are linearly independent",   A.rank() == n),
        ("(i) rows are linearly independent",      A.T.rank() == n),
        ("(j) columns span R^n",                   A.rank() == n),
    ]
    for name, ok in rows:
        print(f"{'yes' if ok else 'no ':4} {name}")
    print("=> invertible:", all(ok for _, ok in rows))
    if A.nullspace():
        print("non-trivial solution of Ax=0:", list(A.nullspace()[0]))

invertibility_report([[1, 2, 3], [0, 1, 4], [1, 3, 7]])      # example 3.21
print()
invertibility_report([[1, 2, 0], [0, 1, 1], [1, 0, -1]])

<div dir="rtl">

כל התנאים תמיד יוצאים יחד "כן" או יחד "לא" – זה בדיוק תוכן המשפט. נבדוק על הרבה מטריצות אקראיות:

</div>

In [ ]:
rng = np.random.default_rng(5)
agree = 0
for _ in range(300):
    M = sp.Matrix(rng.integers(-2, 3, (3, 3)))
    conds = [M.rref()[0] == sp.eye(3), len(M.nullspace()) == 0, M.rank() == 3, M.T.rank() == 3]
    agree += (all(conds) or not any(conds))
print(f"all conditions agree in {agree} of 300 random matrices")

<div dir="rtl">

## 2. תלות בפרמטר

עבור אילו $k$ המטריצה $\begin{bmatrix}1&2\\3&k\end{bmatrix}$ הפיכה?

</div>

In [ ]:
k = sp.symbols("k")
A = sp.Matrix([[1, 2], [3, k]])
print("row-reduce:", A.echelon_form())
for val in [0, 5, 6, 7]:
    print(f"k = {val}: invertible = {A.subs(k, val).rank() == 2}")

<div dir="rtl">

## 3. העתקה לינארית (הגדרה 3.10.7)

בדיקה מספרית של שני התנאים על הרבה וקטורים אקראיים. (בדיקה כזו יכולה **להפריך** לינאריות, אבל לא להוכיח אותה – הוכחה עושים על הנייר.)

</div>

In [ ]:
def looks_linear(T, n, trials=200, seed=0):
    rng = np.random.default_rng(seed)
    for _ in range(trials):
        v, w = rng.normal(size=n), rng.normal(size=n); s = rng.normal()
        if not (np.allclose(T(v + w), T(v) + T(w)) and np.allclose(T(s*v), s*T(v))):
            return False
    return True

T = lambda v: np.array([v[0] + 2*v[1], 3*v[0]])
S = lambda v: np.array([v[0] + 1, v[1]])
U = lambda v: np.array([v[0]*v[1], v[0]])
print("T(x,y) = (x+2y, 3x):", looks_linear(T, 2))
print("S(x,y) = (x+1, y):  ", looks_linear(S, 2))
print("U(x,y) = (xy, x):   ", looks_linear(U, 2))

<div dir="rtl">

## 4. יישום: טרנספורמציות של המישור

נצייר מה כל מטריצה עושה לריבוע היחידה ולרשת קווים. מטריצה **סינגולרית** "מועכת" את המישור לקו – ואת זה אי אפשר לבטל.

</div>

In [ ]:
def plot_map(ax, M, title):
    t = np.linspace(-1, 1, 9)
    for c in t:
        L1 = M @ np.vstack([np.full_like(t, c), t]); L2 = M @ np.vstack([t, np.full_like(t, c)])
        ax.plot(L1[0], L1[1], color="lightgray", lw=.8); ax.plot(L2[0], L2[1], color="lightgray", lw=.8)
    sq = M @ np.array([[0, 1, 1, 0, 0], [0, 0, 1, 1, 0]])
    ax.fill(sq[0], sq[1], alpha=.4)
    e1, e2 = M[:, 0], M[:, 1]
    ax.arrow(0, 0, *e1, color="C1", width=.02); ax.arrow(0, 0, *e2, color="C2", width=.02)
    ax.set_title(f"{title}\ninvertible: {abs(np.linalg.det(M)) > 1e-12}")
    ax.set_aspect("equal"); ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.2, 2.2); ax.grid(True, alpha=.3)

c, s = np.cos(np.pi/6), np.sin(np.pi/6)
maps = {"rotation 30°": np.array([[c, -s], [s, c]]),
        "shear": np.array([[1, 1], [0, 1]]),
        "reflection": np.array([[1, 0], [0, -1]]),
        "projection on x-axis": np.array([[1, 0], [0, 0]]),
        "columns dependent": np.array([[1, 2], [0.5, 1]])}
fig, ax = plt.subplots(1, 5, figsize=(20, 4.5))
for a, (name, M) in zip(ax, maps.items()):
    plot_map(a, M, name)
plt.show()

<div dir="rtl">

## 5. יישום: שכבות לינאריות ברשת נוירונים

בלי פונקציה לא-לינארית בין השכבות, שלוש שכבות שקולות לכפל במטריצה אחת.

</div>

In [ ]:
rng = np.random.default_rng(6)
W1, W2, W3 = rng.normal(size=(5, 4)), rng.normal(size=(3, 5)), rng.normal(size=(2, 3))
x = rng.normal(size=4)
three_layers = W3 @ (W2 @ (W1 @ x))
one_matrix = (W3 @ W2 @ W1) @ x
print(three_layers, one_matrix, np.allclose(three_layers, one_matrix))

relu = lambda z: np.maximum(z, 0)
with_relu = W3 @ relu(W2 @ relu(W1 @ x))
print("with ReLU between layers the result differs:", with_relu)

<div dir="rtl">

## 6. בדקו את עצמכם

</div>

In [ ]:
invertibility_report([[1, 2, 3], [4, 5, 9], [7, 8, 15]])
T3 = sp.Matrix([[1, 0, -1], [0, 2, 0]])
x, y, z = sp.symbols("x y z")
print("T(x,y,z) =", list(T3 * sp.Matrix([x, y, z])))